# Speed planning

**Brake *before* the corner, and find the grip that is really there.**

Unit 6 made the car safe on a sliding car: a cap, v ≤ √(A_LAT / |κ_ahead|), from the sharpest curvature in the next 50 m.
`agents/unit-06.py` finishes every race clean on racekit's carla-like car.
It is also **slow**.
Race it on `test-circuit` (`carla_like(noise=True)`, seed 0) and look at its speed:

In [ ]:
import math
import sys
from pathlib import Path
from tempfile import mkdtemp

import matplotlib.pyplot as plt
import numpy as np
from racekit import write_variant
from racekit.cars import carla_like
from racekit.runner import load_submission, run_race, shim_installed
from racekit.tracks import grip_sections, load_track

AGENTS = Path("../../projects/racekit/agents")
UNIT_06, UNIT_07 = AGENTS / "unit-06.py", AGENTS / "unit-07.py"
workdir = Path(mkdtemp(prefix="roar-unit-07-"))
TICK = 0.05 + 1e-9  # race times are pinned to one control tick
OFFSET_TOL = 0.01 + 1e-9  # m, the book's tolerance on a max |cross-track error|
CLEAN_OFFSET = 2.0  # m: the clean rule's limit


def variant(source, name, **constants):
    # Unit 1's idiom: a copy of a submission with some module constants changed.
    return write_variant(source, workdir / f"{name}.py", {key: repr(value) for key, value in constants.items()})


def max_offset(result):
    # The largest |cross-track error| over the driving rows (held == 0).
    telemetry = result.telemetry
    return float(np.max(np.abs(telemetry["offset"][telemetry["held"] == 0])))


def clean(result):
    # The unit's clean rule: finished, 0 collisions, and never more than 2 m off the line.
    return result.finished and result.collisions == 0 and max_offset(result) <= CLEAN_OFFSET


def summary(result):
    if clean(result):
        return f"clean, {result.total_time:6.2f} s, max |offset| {max_offset(result):.2f} m"
    finished = "finished" if result.finished else "did not finish"
    return f"NOT clean: {result.collisions} collisions, {finished}, max |offset| {max_offset(result):.2f} m"


def race(source, seed=0, max_seconds=600.0, track="test-circuit"):
    # Every race of this lesson: the carla-like car with its noise, one seed.
    return run_race(source, track, car=carla_like(noise=True), seed=seed, max_seconds=max_seconds)


unit_06 = race(UNIT_06, max_seconds=1500.0)
driving = unit_06.telemetry["held"] == 0
top_speed = float(np.max(unit_06.telemetry["v"][driving]))
print("Unit 6 on test-circuit, seed 0:", summary(unit_06))
print(f"its top speed: {top_speed:.2f} m/s")
assert clean(unit_06) and abs(unit_06.total_time - 422.55) <= TICK
assert top_speed < 20.5

fig, ax = plt.subplots(figsize=(11, 3.5))
early = unit_06.telemetry["t"] < 150.0
ax.plot(unit_06.telemetry["t"][early], unit_06.telemetry["v"][early])
ax.set_xlabel("t (s)")
ax.set_ylabel("speed (m/s)")
ax.set_title("Unit 6 on test-circuit: the first lap and a half")
plt.show()

Two things are wrong with that trace, and neither is a crash.

- **It never goes faster than about 20 m/s** (20.40 m/s at most, the speed loop's overshoot), even on the longest straight of the track, because Unit 6's goal is capped at `TARGET_SPEED = 20`.
  Raise the cap and the agent brakes too late: the corner enters its 50 m window only when the car is already close, and at 45 m/s braking to 10 m/s takes about 190 m at 5 m/s².
- **It brakes when the corner is in the window, not when it has to.**
  The cap is a reaction to the curvature ahead, not a plan.

Over seeds 0–4 Unit 6 takes a median of **422.65 s** (probe record, plan `roar-006` U6.4; Unit 6's milestone races it).
This unit plans the whole lap's speed **once**, before the race starts, so that the car is at full speed wherever it can be and starts braking exactly where it must.
Then it asks how hard the plan may push, section by section, and how to know when it has pushed too hard.

## 1. The forward–backward speed profile

Number the waypoints i = 0 … n − 1 around the lap, let Δs_i be the length of the segment from i to i + 1, and κ_i the Menger curvature at i (Unit 2).
The profile is built in three steps.

1. **The cap of every waypoint**, Unit 6's limit applied where the corner is:
   $$\bar v_i = \min\left(v_{\text{top}},\ \sqrt{A_{\text{LAT}} / |\kappa_i|}\right).$$
2. **The backward pass (braking).** A car that must be at v_{i+1} at waypoint i + 1, and can brake at A_DEC, can be at most
   $$v_i \le \sqrt{v_{i+1}^2 + 2\,A_{\text{DEC}}\,\Delta s_i}$$
   at waypoint i (constant deceleration: v² changes by 2·a·Δs).
   Walk the lap **backwards**, lowering every v_i that breaks this: the slow speed of a corner spreads back along the straight before it, as a braking ramp.
3. **The forward pass (accelerating)**, the same rule forwards with A_ACC: v_{i+1} ≤ √(v_i² + 2·A_ACC·Δs_i).
   A car leaving a corner cannot reach full speed at once.

The lap is a **loop**: waypoint n − 1 brakes for waypoint 0's corner, and the backward pass reaches n − 1 before it has lowered 0.
So each pass goes **twice** around the lap; the second time round, the speeds at the wrap-around have settled.

Here are the passes, on `test-circuit`, with the reference's numbers (A_LAT = 4.6 m/s², A_DEC = 5.0 m/s², A_ACC = 4.0 m/s², v_top = 45 m/s), checked against the profile `agents/unit-07.py` builds in its `__init__`:

In [ ]:
TOP_SPEED, A_LAT, A_DEC, A_ACC = 45.0, 4.6, 5.0, 4.0
track = load_track("test-circuit")
xs, ys, n = track.x, track.y, len(track.x)
before, after = (np.arange(n) - 1) % n, (np.arange(n) + 1) % n
twice_area = (xs - xs[before]) * (ys[after] - ys[before]) - (ys - ys[before]) * (xs[after] - xs[before])
sides = (np.hypot(xs - xs[before], ys - ys[before]) * np.hypot(xs[after] - xs, ys[after] - ys)
         * np.hypot(xs[after] - xs[before], ys[after] - ys[before]))
kappa = 2.0 * twice_area / sides  # Unit 2's Menger curvature, cyclic
ds = np.hypot(xs[after] - xs, ys[after] - ys)  # segment i -> i + 1 (the last one closes the lap)
cap = np.array([min(TOP_SPEED, math.sqrt(A_LAT / abs(k))) if abs(k) > 1e-9 else TOP_SPEED for k in kappa])


def backward_pass(v, a_dec, laps):
    v = list(v)
    for _ in range(laps):
        for i in range(n - 1, -1, -1):
            v[i] = min(v[i], math.sqrt(v[(i + 1) % n] ** 2 + 2.0 * a_dec * ds[i]))
    return np.array(v)


def forward_pass(v, a_acc, laps):
    v = list(v)
    for _ in range(laps):
        for i in range(n):
            v[(i + 1) % n] = min(v[(i + 1) % n], math.sqrt(v[i] ** 2 + 2.0 * a_acc * ds[i]))
    return np.array(v)


profile = forward_pass(backward_pass(cap, A_DEC, 2), A_ACC, 2)
once = forward_pass(backward_pass(cap, A_DEC, 1), A_ACC, 1)
thrice = forward_pass(backward_pass(cap, A_DEC, 3), A_ACC, 3)
print(f"waypoints changed by the second time round: {int(np.sum(np.abs(profile - once) > 1e-9))}, "
      f"by a third: {int(np.sum(np.abs(thrice - profile) > 1e-9))}")

with shim_installed():
    Unit07, module_name = load_submission(UNIT_07)
agent = Unit07(track.waypoints, None)  # __init__ plans the profile from the waypoints alone
print(f"largest difference from the agent's profile: {np.max(np.abs(profile - np.array(agent.profile))):.1e} m/s")
assert np.max(np.abs(profile - np.array(agent.profile))) < 1e-9
assert np.all(profile <= cap + 1e-12)
assert np.all(profile ** 2 <= profile[after] ** 2 + 2.0 * A_DEC * ds + 1e-9)  # every braking ramp is feasible
assert np.max(np.abs(thrice - profile)) == 0.0
print(f"profile: {profile.min():.2f} to {profile.max():.2f} m/s")

The second time round changes 26 waypoints, just after the start line: the lap's last corner is just before the line, and the car is still accelerating out of it there, which the forward pass only learns once it has gone round once.
A third time changes nothing.

Now put the plan next to Unit 6's goal.
Unit 6's goal at waypoint i is min(20, √(6 / |κ_ahead|)), with κ_ahead the sharpest curvature over waypoints i … i + 24 (`A_LAT = 6` there).

In [ ]:
kappa_ahead = np.array([np.max(np.abs(kappa[np.arange(i, i + 25) % n])) for i in range(n)])
unit_6_goal = np.array([min(20.0, math.sqrt(6.0 / k)) if k > 1e-9 else 20.0 for k in kappa_ahead])
print(f"the planned lap (the sum of ds / mean speed): {np.sum(ds / ((profile + profile[after]) / 2.0)):.1f} s; "
      f"Unit 6's goal as a plan: {np.sum(ds / ((unit_6_goal + unit_6_goal[after]) / 2.0)):.1f} s")

fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(track.s, cap, color="lightgrey", linewidth=1.0, label="the cap, sqrt(A_LAT / |kappa|), at most 45")
ax.plot(track.s, profile, label="Unit 7: the forward-backward profile")
ax.plot(track.s, unit_6_goal, label="Unit 6: min(20, sqrt(6 / |kappa_ahead|))")
ax.set_xlabel("distance along the lap (m)")
ax.set_ylabel("speed (m/s)")
ax.set_title("test-circuit: the planned speed of every waypoint")
ax.legend()
plt.show()

Read the plot from left to right.
On every straight the profile climbs at A_ACC to 45 m/s, more than twice Unit 6's ceiling.
Before every corner it comes down along a **ramp**, the braking pass, and it reaches the corner's cap (the grey dips) exactly at the corner.
Unit 6's goal drops in a step, 50 m before the corner, and to a lower speed: its A_LAT of 6 applies to the sharpest point anywhere in the window, so it is low for the whole window.

The profile is a **plan, not a promise**.
It assumes the car follows it exactly, at exactly the planned rates; the next section races it and finds out what that assumption costs.

## 2. Racing the profile at 45 m/s

`agents/unit-07.py` is Unit 6's file with `SPEED_MODE = "profile"`: in `step()`, the speed goal is the profile instead of Unit 6's cap, fed to the same PI speed loop, with `TARGET_SPEED = 45`.
Two more things had to change before it was clean, and both are about **looking further ahead at speed**.

**The steering lookahead.**
Unit 5's pure pursuit looks `PP_K` = 0.3 s ahead: 6 m at 20 m/s, 13.5 m at 45 m/s.
That was the right amount at 20 m/s; at 45 m/s it is not.
Race the reference with `PP_K = 0.3` and with its own 0.5:

In [ ]:
reference = race(UNIT_07)
short_lookahead = race(variant(UNIT_07, "pp-k-0.3", PP_K=0.3))
print("PP_K = 0.3:", summary(short_lookahead))
print("PP_K = 0.5:", summary(reference))
assert short_lookahead.collisions == 10 and not short_lookahead.finished  # U7.1, seed 0
assert clean(reference) and abs(reference.total_time - 304.15) <= TICK

The short lookahead hits the wall 10 times in 600 s; the reference is clean, in **304.15 s**.
Over seeds 0–4 (probe record, plan `roar-006` U7.1; the milestone and a routed test re-race it):

| `PP_K` (s) | 0.3 | 0.4 | 0.5 | 0.6 | 0.7 |
|---|---|---|---|---|---|
| collisions per seed | [10, 9, 8, 9, 10] | [6, 6, 6, 6, 6] | clean | clean, 304.20 s | clean, 304.20 s |

0.5 is the smallest clean value, and the longer lookaheads are no faster beyond the noise, so the reference keeps 0.5.
It is Unit 5's lesson again at a new speed: the lookahead that keeps the steering stable grows with the speed and the delay.

**The speed lead.**
The PI speed loop does not follow its goal instantly: a step down in the goal takes a while to become brake.
If the goal is the profile **at the car**, the loop starts braking when the car is already on the ramp, arrives above the profile, and enters the corner too fast.
So the goal is read **ahead**: the profile at the first waypoint at least max(`PROFILE_LEAD_S`·v, 2 m) of path ahead of the car.
The lead is measured in **seconds of travel**, because the loop's lag is a time: 0.3 s is 13.5 m at 45 m/s and 3 m at 10 m/s, on any track and any waypoint spacing (a lead of "two waypoints" would mean something different on CARLA's map).

Well inside the edge the lead barely matters: at (A_DEC 4.0, A_LAT 4.6) every lead from 0.1 s to 0.35 s is clean, within 0.25 s of each other.
It matters **at the edge**: here is the more aggressive setting A_DEC = 4.0, A_LAT = 5.0, with a lead of 0.15 s and of 0.3 s.

In [ ]:
edge_short_lead = race(variant(UNIT_07, "edge-lead-0.15", A_DEC=4.0, A_LAT=5.0, PROFILE_LEAD_S=0.15))
edge_lead = race(variant(UNIT_07, "edge-lead-0.3", A_DEC=4.0, A_LAT=5.0))
print("A_DEC 4.0, A_LAT 5.0, lead 0.15 s:", summary(edge_short_lead))
print("A_DEC 4.0, A_LAT 5.0, lead 0.30 s:", summary(edge_lead))
assert edge_short_lead.collisions == 6 and not clean(edge_short_lead)  # U7.2, seed 0
assert clean(edge_lead) and abs(edge_lead.total_time - 303.25) <= TICK
assert abs(max_offset(edge_lead) - 1.55) <= OFFSET_TOL

# the reference's speed against its own plan: the profile at the car's waypoint (racekit's progress p is
# waypoint p + 1), over 40 s of the second lap
telemetry = reference.telemetry
window = (telemetry["t"] > 120.0) & (telemetry["t"] < 160.0)
fig, ax = plt.subplots(figsize=(11, 3.5))
ax.plot(telemetry["t"][window], profile[(telemetry["progress"][window] + 1) % n], label="the profile at the car")
ax.plot(telemetry["t"][window], telemetry["v"][window], label="the car's speed")
ax.set_xlabel("t (s)")
ax.set_ylabel("speed (m/s)")
ax.set_title("agents/unit-07.py on test-circuit: following the plan, with a 0.3 s lead")
ax.legend()
plt.show()

With the short lead the car collides 6 times; with 0.3 s it is clean, 303.25 s.
The plot shows why a lead is needed at all: on every ramp the car's speed follows the profile **late**, and the lead is what makes "late" land before the corner instead of in it.

The lead study over seeds 0–4 (probe record, plan `roar-006` U7.2; the rows' 0.15 s and 0.3 s cells are asserted by a routed test):

| (A_DEC, A_LAT) | lead 0.1 s | 0.15 s | 0.2 s | 0.25 s | 0.3 s | 0.35 s |
|---|---|---|---|---|---|---|
| (4.0, 4.6) | clean | clean | clean | clean | clean | clean |
| (4.0, 5.0) | collides | collides | clean | clean | clean | clean |
| (6.0, 4.0) | collides | collides | over 2 m | clean | clean | clean |

At (4.0, 4.6) every lead is clean, 309.70–309.95 s, and the max |offset| falls from 1.53 m to 1.25 m as the lead grows; at (6.0, 4.0) a 0.2 s lead finishes without a collision but more than 2 m off the line (2.03–2.36 m).
`PROFILE_LEAD_S = 0.3` is inside the clean range everywhere it was probed, and the book fixes it there; an exercise sweeps it.

## 3. Two knobs: how hard may the plan push?

The profile has two numbers that decide how fast it is: **A_LAT**, how much cornering it plans, and **A_DEC**, how hard it plans to brake.
Unit 6 found the cap's A_LAT edge alone.
Here the two act together: a planner that brakes later (higher A_DEC) arrives at the corner with less room for error, so it can afford less cornering.

The grid, on `test-circuit` with the carla-like car, lead 0.3 s, median race times over seeds 0–4 (probe record, plan `roar-006` U7.3; "no" = not clean on any seed; each row's last clean cell and first unclean cell are asserted by a routed test):

| A_DEC \ A_LAT | 4.0 | 4.4 | 4.6 | 4.8 | 5.0 | 5.2 | 5.4 | 5.6 |
|---|---|---|---|---|---|---|---|---|
| 3.0 | 329.75 | 322.05 | 318.60 | 315.05 | 311.70 | 308.45 | no | no |
| 4.0 | 320.50 | 313.25 | 309.80 | 306.55 | 303.25 | no | no | no |
| 4.5 | 317.30 | 310.05 | 306.70 | 303.40 | 4/5 | no | no | no |
| 5.0 | 314.65 | 307.55 | 304.15 | **300.90** | no | no | no | no |
| 6.0 | 311.40 | 304.15 | no | no | no | no | no | no |

((4.5, 5.0) is clean on 4 of 5 seeds: seed 1 collides once.)

Two patterns.

- **Within a row**, more A_LAT is faster until it is not clean: the edge.
- **Down the columns**, the edge falls as A_DEC rises: 5.2, 5.0, 4.8, 4.8, 4.4.
  The fastest clean cell is (A_DEC 5.0, A_LAT 4.8), 300.90 s.

Why does braking harder cost cornering?
Part of the answer is the **car's friction circle** (Unit 6): a tyre that is braking has less grip left for cornering, so braking late, into the corner, leaves less for the corner.
But that is not the whole answer, because **the planner does not model the circle**: its cornering cap and its braking pass are two separate approximations, each using its whole budget.
The speed loop arriving above the profile (§2), and pure pursuit cutting the corner on a line that is not the waypoints' line (Unit 5), move the edge too.
An exercise checks the circle directly: a_lat² + a_dec² along the edge is **not** constant.

**The margin rule.**
The reference ships **(A_DEC 5.0, A_LAT 4.6)**, one grid step inside the fastest clean edge, not the edge itself: five clean seeds do not prove the sixth, and CARLA's grip is not racekit's.
Over ten seeds it takes **304.15 s** (spread 0.20), all clean (probe record, plan `roar-006` U7.4; the milestone races seeds 0–4).

In [ ]:
unit_6_seed_0, unit_7_seed_0 = unit_06.total_time, reference.total_time
print(f"seed 0: Unit 6 {unit_6_seed_0:.2f} s, Unit 7 {unit_7_seed_0:.2f} s, "
      f"a gain of {unit_6_seed_0 - unit_7_seed_0:.2f} s ({(unit_6_seed_0 - unit_7_seed_0) / unit_6_seed_0:.0%})")
assert clean(unit_06) and clean(reference) and unit_6_seed_0 - unit_7_seed_0 > 100.0

fig, ax = plt.subplots(figsize=(11, 3.5))
for name, result in (("Unit 6", unit_06), ("Unit 7", reference)):
    early = result.telemetry["t"] < 150.0
    ax.plot(result.telemetry["t"][early], result.telemetry["v"][early], label=name)
ax.set_xlabel("t (s)")
ax.set_ylabel("speed (m/s)")
ax.set_title("test-circuit, seed 0: the cap against the plan")
ax.legend()
plt.show()

The gain over Unit 6 holds on every track, and far beyond the noise (medians over seeds 0–4, all clean; probe record, plan `roar-006` U7.5, asserted by a routed test):

| track | Unit 6 | Unit 7 |
|---|---|---|
| `oval` | 216.50 s (0.10) | 152.75 s (0.10) |
| `test-circuit` | 422.65 s (0.30) | 304.15 s (0.20) |
| `monza-like` | 880.50 s (0.15) | 515.10 s (0.15) |

A large median gain at 0 collisions: the unit's first gate.

Some numbers in the file were **not** tuned: `A_ACC = 4.0`, `LEAD_MIN_M = 2.0`, the lead's 0.3 s and the speed loop's gains are fixed at the probe's values.
Tuning many knobs at once is Unit 12's job.

## 4. Per-section knobs

One A_LAT for the whole lap is a compromise: it is set by the section where the plan is most wrong.
Elsewhere the car might take more.
`agents/unit-07.py` has one **knob** per section, `GRIP`, that multiplies A_LAT in that section's caps.
racekit's grip sections cut the lap into six equal pieces of arc length from waypoint 0 (`grip_sections`); the agent computes the same map from its waypoints:

In [ ]:
sections = grip_sections(track)
assert agent.section == sections.tolist()  # the agent's map is racekit's (U7.9)
print("waypoints per section:", [int(np.sum(sections == k)) for k in range(6)])

fig, ax = plt.subplots(figsize=(6, 5))
points = ax.scatter(xs, ys, c=sections, s=4, cmap="viridis")
fig.colorbar(points, ax=ax, label="grip section")
ax.plot([xs[0]], [ys[0]], "ks", label="waypoint 0, the start")
ax.set_aspect("equal")
ax.set_title("test-circuit's six grip sections")
ax.legend()
plt.show()

**One knob at a time.**
Raise one section's knob from 1.0 up the grid (1.05, 1.1, 1.15, 1.2, 1.3, 1.4, 1.5), the others at 1.0, five seeds per step, and keep the last clean step.
Every section has room (probe record, plan `roar-006` U7.6; each kept step is asserted clean on seeds 0–4 by a routed test):

| section | 0 | 1 | 2 | 3 | 4 | 5 |
|---|---|---|---|---|---|---|
| clean alone up to | 1.2 | 1.5 (the grid's top) | 1.1 | 1.1 | 1.05 | 1.1 |

**The stacking trap.**
So set all six at once, (1.2, 1.5, 1.1, 1.1, 1.05, 1.1)?

In [ ]:
STACK = (1.2, 1.5, 1.1, 1.1, 1.05, 1.1)
stacked = race(variant(UNIT_07, "stack", GRIP=STACK))
print("the stacked single-knob results, seed 0:", summary(stacked))
assert stacked.collisions == 23 and not clean(stacked)  # U7.6, seed 0

23 collisions, and the plan's record has 23 on **every one of seeds 0–9**.
Each knob was found with the others at 1.0.
A faster section 0 hands section 1 a car that arrives faster, a different starting speed and a different line; the knobs are not independent, and the single-knob results do not add.

**The sequential pass.**
Sweep section 0 and **keep** its knob; then sweep section 1 **on top of it**; and so on, each step on seeds 0–4, each section stopping at its first unclean step (probe record, plan `roar-006` U7.7; a routed test asserts every row):

| section | kept | the first unclean step |
|---|---|---|
| 0 | 1.2, 303.10 s | 1.3: collisions [9, 0, 0, 4, 13] |
| 1 | 1.4, 298.35 s | 1.5: [23, 23, 23, 23, 23] |
| 2, 4, 5 | 1.0 | 1.05: seed 4 collides 18 times, seeds 0–3 clean |
| 3 | 1.0 | 1.05: [0, 18, 0, 0, 18] |

Section 1 was clean **alone** at 1.5 and collides at 1.5 on top of section 0's 1.2: the interaction, measured.

**The ten-seed check.**
The pass ends at (1.2, 1.4, 1.0, 1.0, 1.0, 1.0), clean on seeds 0–4.
Sections 2–5 each failed their first step on seed 4, so the vector sits right **on** an edge.
Race it on a seed it was never tuned on:

In [ ]:
EDGE_VECTOR = (1.2, 1.4, 1.0, 1.0, 1.0, 1.0)
edge_seed_5 = race(variant(UNIT_07, "edge-vector", GRIP=EDGE_VECTOR), seed=5)
print("the sequential pass's result, seed 5:", summary(edge_seed_5))
assert edge_seed_5.collisions == 18 and not clean(edge_seed_5)  # U7.7: seeds 5 and 8 fail
assert abs(max_offset(edge_seed_5) - 6.48) <= OFFSET_TOL

ACCEPTED = (1.15, 1.3, 1.0, 1.0, 1.0, 1.0)
accepted = race(variant(UNIT_07, "accepted", GRIP=ACCEPTED))
print("one step back on the raised knobs, seed 0:", summary(accepted))
assert clean(accepted) and abs(accepted.total_time - 299.65) <= TICK

Seed 5 collides 18 times, 6.48 m off the line; over seeds 0–9 the vector fails on seeds 5 and 8 and is clean on the other eight.
**Five seeds were not enough** for a result that sits on an edge.
The margin rule fixes it: step **every raised knob back once**, to (1.15, 1.3, 1.0, 1.0, 1.0, 1.0), and check again on ten seeds.
That vector is clean on all ten, **299.60 s** (spread 0.15) (probe record, plan `roar-006` U7.7; the milestone races seeds 0–4).

**Like with like.**
Do the knobs beat a single global setting?
The fair comparison is against the **best** global setting, not just the cautious reference (ten seeds each, all clean; probe record, plan `roar-006` U7.8, asserted by a routed test):

| setting | median (spread) | margin? |
|---|---|---|
| the knobs (1.15, 1.3, 1, 1, 1, 1) at (5.0, 4.6) | **299.60 s (0.15)** | yes, one step back |
| the reference (5.0, 4.6) | 304.15 s (0.20) | yes, one step inside |
| the fastest clean global setting (5.0, 4.8) | 300.93 s (0.20) | none: it is the edge |

The knobs beat both beyond the noise: against the reference by 4.55 s, and against the edge by 1.33 s, more than either spread, while keeping a margin the edge does not have.
That is the unit's second gate.

**The knobs are not μ.**
A knob of 1.3 in section 1 does **not** mean the tyres there grip 30% more than A_LAT assumes.
The knob multiplies the planner's A_LAT, so it absorbs everything the planner gets wrong in that section: the grip, the noise in the waypoints' curvature, the line pure pursuit actually drives, braking and cornering combined, and the speed loop's lag behind the lead.
That is why they are tuned **per track** (a section is a sixth of *this* lap), why `agents/unit-07.py` ships them all at 1.0, and why μ itself waits for Unit 8, which identifies the car's grip from its logs.

## 5. To the real car

Everything above is a model number on racekit's carla-like car.
The lab of this unit (`labs/real-sim.ipynb`) races `agents/unit-07.py` on CARLA's Monza, **exactly as shipped**, records it with a section map (a sidecar file next to the telemetry), and reports each section's time, speed, peak cross-track error and collisions against racekit's prediction.
If that first run is clean, it runs the sequential pass on the real car, on a coarse grid and a fixed budget of runs, and re-races the result before trusting it.

The tools of this unit carry over to every later one:

- a **plan** for the whole lap, built from the track before the race starts;
- a **clean rule** that is stricter than "no crash";
- **one knob at a time**, kept and stacked in order, never assembled from separate sweeps;
- a result **checked on seeds it was not tuned on**, and a **margin** when it sits on an edge.

Next unit: system identification, where the car's μ, delay and actuator maps are measured instead of guessed.